# Gain & AGC Demo
Replicate Seismic Unix processing pipelines (like `sugain` and `suxwigb` clipping).

In [ ]:
%pip install bokeh segyio

# *** Install pyseiskit from PyPI
# %pip install pyseiskit
# *** Install pyseiskit for local development
# %pip install -e ..
!uv pip install -e .. --reinstall

## 1A. Option 1: Create Simple Synthetic Data
Run this cell to generate a small 2D array representing 5 seismic traces, each with 100 time samples.

In [ ]:
import numpy as np

# 100 samples, 5 traces
gatherData = np.random.randn(100, 5) 
timeSamples = np.arange(100)
traceOffsets = np.arange(5, dtype=float)


## 1B. Option 2: Load Real `.su` or `.sgy` Data
Run this cell instead if you have the file `tac-204RL239.su` in the same folder as this notebook.

In [ ]:
import segyio
from seismicReader import readSeismicFile

filename = 'tac-204RL239.su'

# We only want to plot a single gather to avoid rendering thousands of traces.
# FieldRecord corresponds to the 'fldr' header in SU/SEGY.
gatherKey = segyio.TraceField.FieldRecord
gatherIndex = 50  # Change this to a valid fldr number from your dataset!

try:
    gatherData, traceOffsets, timeSamples = readSeismicFile(filename, gatherKey=gatherKey, gatherIndex=gatherIndex)
    print(f"Loaded {gatherData.shape[1]} traces for fldr {gatherIndex} with {gatherData.shape[0]} samples each.")
except FileNotFoundError:
    print(f"File '{filename}' not found. Please place it in the 'demos' folder or stick to Option 1A.")
except ValueError as e:
    print(f"Error: {e}")

## 2. Process Data for Wiggles
Process the seismic data and compute wiggle trace geometries.

In [ ]:
from pyseiskit import sourceData
from pyseiskit import gain
from pyseiskit import clip

# Replicating SU pipeline: suwind | sugain agc=1 wagc=1.0 | suxwigb perc=99
sampleIntervalSeconds = timeSamples[1] - timeSamples[0]

# 1. Apply AGC (Optional, just like sugain agc=1 wagc=1.0)
gainedGatherData = gain.applyAGC(gatherData, wagc=1.0, intervalTimeSamples=sampleIntervalSeconds)
# gainedGatherData = gatherData  # uncomment above to apply AGC

# 2. Apply Percentile Clipping (Like suxwigb perc=99 flat-topping)
clippedGatherData = clip.applyPercentileClip(gainedGatherData, percentile=99.0)

# 3. Scale physical dimensions (overlap=1.0 is SU default xcur/overlap)
scaledGatherData = sourceData.rescaleDataForWiggle(clippedGatherData, traceOffsets, overlap=1.0)

# 4. Generate geometry
lineData = sourceData.wiggleLinesDataFactory(scaledGatherData, traceOffsets, timeSamples)
patchData = sourceData.wigglePatchesDataFactory(scaledGatherData, traceOffsets, timeSamples, fill_mode='positive')

## 3. Visualize Results
Here we use Bokeh to visualize the gained results interactively.

In [ ]:
from bokeh.plotting import figure, show, output_notebook
output_notebook()

bokehPlot = figure(width=600, height=400, y_range=(timeSamples[-1], timeSamples[0]))
bokehPlot.multi_line(**lineData, color='black', line_width=0.5)
bokehPlot.patches(**patchData, color='black', line_width=0)

show(bokehPlot)